In [ ]:
# ========================= SENSITIVITY ANALYSIS — KD-CMAES Student (EfficientNet-B0 + CBAM) =========================
# Sweeps: Temperature (T), KD alpha (CE:KD balance), Learning Rate (lr)
# These are exactly the 3 params your CMA-ES search tunes — so this sweep shows WHY those values matter.
# Teacher ensemble (ResNet50 + DenseNet121) is trained ONCE and reused across every sweep run
# (only the lightweight student is retrained per setting, for short epochs, which is what's cheap to sweep).

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "/kaggle/input/datasets/sabbir4724/liver-histopathology"
OUT_DIR = "/kaggle/working/sensitivity_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
TEACHER_EPOCHS = 5
TEACHER_LR = 1e-4
SWEEP_EPOCHS = 5          # short training budget per config, identical across all runs for fair comparison

# Defaults held fixed while sweeping the other two.
# Edit these to your CMA-ES best (alpha, temperature, lr) if you already have them.
DEFAULT_ALPHA = 0.5
DEFAULT_T     = 5.0
DEFAULT_LR    = 1e-4

T_VALUES     = [1, 2, 3, 4, 5, 6, 8, 10]
ALPHA_VALUES = [0.1, 0.3, 0.5, 0.7, 0.9]
LR_VALUES    = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3]

ORDINAL_LAMBDA = 0.3
FOCAL_GAMMA = 2.0

# ========================= DATA (same 70/15/15 split as your original notebook) =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

targets = np.array(full_dataset.targets)
train_idx, temp_idx = train_test_split(np.arange(len(targets)), test_size=0.30, stratify=targets, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=SEED)

train_dataset = torch.utils.data.Subset(datasets.ImageFolder(DATA_DIR, transform=train_transform), train_idx)
val_dataset = torch.utils.data.Subset(full_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


def compute_class_weights(target_array, num_classes):
    counts = np.bincount(target_array, minlength=num_classes)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


CLASS_WEIGHTS = compute_class_weights(targets[train_idx], NUM_CLASSES).to(DEVICE)

# ========================= DOMAIN-AWARE LOSS (focal + ordinal, F0<F1<F2<F3<F4) =========================
class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def ordinal_distance_penalty(logits, targets, num_classes):
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(num_classes, device=logits.device).float()
    expected_idx = (probs * class_indices).sum(dim=1)
    return F.mse_loss(expected_idx, targets.float())


focal_loss_fn = FocalLoss(class_weights=CLASS_WEIGHTS, gamma=FOCAL_GAMMA)


def domain_loss(logits, targets, num_classes):
    return focal_loss_fn(logits, targets) + ORDINAL_LAMBDA * ordinal_distance_penalty(logits, targets, num_classes)


# ========================= SE + CBAM attention =========================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


# ========================= TEACHER ENSEMBLE (trained once, reused across every sweep run) =========================
def build_teacher(name, num_classes):
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model


def train_one_teacher(model, epochs=5, lr=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        print(f"  [Teacher {model.__class__.__name__}] Epoch {epoch+1}/{epochs} val_acc={correct/total:.4f}")
    return model


class TeacherEnsemble(nn.Module):
    def __init__(self, teachers, weights=None):
        super().__init__()
        self.teachers = nn.ModuleList(teachers)
        for t in self.teachers:
            t.eval()
            for p in t.parameters():
                p.requires_grad = False
        n = len(teachers)
        self.weights = weights if weights is not None else [1.0 / n] * n

    @torch.no_grad()
    def forward(self, x):
        probs = 0
        for w, t in zip(self.weights, self.teachers):
            probs = probs + w * F.softmax(t(x), dim=1)
        return probs


print("\nTraining teacher ensemble (once)...")
teacher_ckpt = "/kaggle/working/teacher_resnet50.pth"
teacher2_ckpt = "/kaggle/working/teacher_densenet121.pth"

teacher1 = build_teacher("resnet50", NUM_CLASSES).to(DEVICE)
if os.path.exists(teacher_ckpt):
    teacher1.load_state_dict(torch.load(teacher_ckpt, map_location=DEVICE))
    print("  Loaded cached teacher1")
else:
    teacher1 = train_one_teacher(teacher1, epochs=TEACHER_EPOCHS, lr=TEACHER_LR)
    torch.save(teacher1.state_dict(), teacher_ckpt)

teacher2 = build_teacher("densenet121", NUM_CLASSES).to(DEVICE)
if os.path.exists(teacher2_ckpt):
    teacher2.load_state_dict(torch.load(teacher2_ckpt, map_location=DEVICE))
    print("  Loaded cached teacher2")
else:
    teacher2 = train_one_teacher(teacher2, epochs=TEACHER_EPOCHS, lr=TEACHER_LR)
    torch.save(teacher2.state_dict(), teacher2_ckpt)

teacher_ensemble = TeacherEnsemble([teacher1, teacher2]).to(DEVICE)
print("✔ Teacher ensemble ready\n")


# ========================= STUDENT (EfficientNet-B0 + CBAM) =========================
class StudentModel(nn.Module):
    def __init__(self, num_classes, use_cbam=True):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.attention = CBAM(1280) if use_cbam else SEBlock(1280)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(self.dropout(x))


def distillation_loss(student_logits, teacher_probs, true_labels, alpha, temperature):
    ce_loss = domain_loss(student_logits, true_labels, NUM_CLASSES)
    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    teacher_soft_T = F.softmax(torch.log(teacher_probs + 1e-8) / temperature, dim=1)
    kd_loss = F.kl_div(student_log_soft, teacher_soft_T, reduction="batchmean") * (temperature ** 2)
    return alpha * kd_loss + (1 - alpha) * ce_loss


# ========================= ONE TRAINING RUN FOR A GIVEN (alpha, T, lr) =========================
def run_config(alpha, temperature, lr, epochs=SWEEP_EPOCHS, tag=""):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    student = StudentModel(NUM_CLASSES, use_cbam=True).to(DEVICE)
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    best_acc, best_f1, best_val_loss = 0.0, 0.0, float("inf")

    for epoch in range(epochs):
        student.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            student_logits = student(x)
            with torch.no_grad():
                teacher_probs = teacher_ensemble(x)
            loss = distillation_loss(student_logits, teacher_probs, y, alpha, temperature)
            loss.backward()
            optimizer.step()

        student.eval()
        preds, gts, val_loss_sum = [], [], 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                student_logits = student(x)
                teacher_probs = teacher_ensemble(x)
                val_loss_sum += distillation_loss(student_logits, teacher_probs, y, alpha, temperature).item()
                preds.extend(student_logits.argmax(1).cpu().numpy())
                gts.extend(y.cpu().numpy())

        val_acc = accuracy_score(gts, preds) * 100
        _, _, val_f1, _ = precision_recall_fscore_support(gts, preds, average="weighted", zero_division=0)
        val_loss = val_loss_sum / len(val_loader)

        if val_acc > best_acc:
            best_acc, best_f1, best_val_loss = val_acc, val_f1, val_loss

        print(f"  [{tag}] Epoch {epoch+1}/{epochs} | Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}% | Val F1(weighted): {val_f1:.4f}")

    return {"best_val_acc": best_acc, "best_val_f1": best_f1, "best_val_loss": best_val_loss}


# ========================= SWEEP 1: TEMPERATURE =========================
print("\n" + "=" * 70)
print("SWEEP 1/3: TEMPERATURE (T)  |  alpha =", DEFAULT_ALPHA, "lr =", DEFAULT_LR)
print("=" * 70)
temp_results = []
for T in T_VALUES:
    print(f"\n--- T = {T} ---")
    res = run_config(alpha=DEFAULT_ALPHA, temperature=T, lr=DEFAULT_LR, tag=f"T={T}")
    temp_results.append({"Temperature": T, **res})
temp_df = pd.DataFrame(temp_results)
temp_df.to_csv(f"{OUT_DIR}/sensitivity_temperature.csv", index=False)
print("\nTemperature Sweep Results:")
print(temp_df.to_string(index=False))

# ========================= SWEEP 2: ALPHA (CE:KD balance) =========================
print("\n" + "=" * 70)
print("SWEEP 2/3: ALPHA (CE:KD balance)  |  T =", DEFAULT_T, "lr =", DEFAULT_LR)
print("=" * 70)
alpha_results = []
for a in ALPHA_VALUES:
    print(f"\n--- alpha = {a} ---")
    res = run_config(alpha=a, temperature=DEFAULT_T, lr=DEFAULT_LR, tag=f"alpha={a}")
    alpha_results.append({"Alpha": a, **res})
alpha_df = pd.DataFrame(alpha_results)
alpha_df.to_csv(f"{OUT_DIR}/sensitivity_alpha.csv", index=False)
print("\nAlpha Sweep Results:")
print(alpha_df.to_string(index=False))

# ========================= SWEEP 3: LEARNING RATE =========================
print("\n" + "=" * 70)
print("SWEEP 3/3: LEARNING RATE (lr)  |  alpha =", DEFAULT_ALPHA, "T =", DEFAULT_T)
print("=" * 70)
lr_results = []
for lr in LR_VALUES:
    print(f"\n--- lr = {lr} ---")
    res = run_config(alpha=DEFAULT_ALPHA, temperature=DEFAULT_T, lr=lr, tag=f"lr={lr}")
    lr_results.append({"LR": lr, **res})
lr_df = pd.DataFrame(lr_results)
lr_df.to_csv(f"{OUT_DIR}/sensitivity_lr.csv", index=False)
print("\nLearning Rate Sweep Results:")
print(lr_df.to_string(index=False))

# ========================= PLOTS =========================
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

ax = axes[0]
ax.plot(temp_df["Temperature"], temp_df["best_val_acc"], "o-", color="#4363d8", linewidth=2, label="Val Accuracy (%)")
ax2 = ax.twinx()
ax2.plot(temp_df["Temperature"], temp_df["best_val_f1"] * 100, "s--", color="#e6194B", linewidth=2, label="Val F1 (weighted, %)")
ax.set_xlabel("Temperature (T)"); ax.set_ylabel("Val Accuracy (%)", color="#4363d8")
ax2.set_ylabel("Val F1 weighted (%)", color="#e6194B")
ax.set_title("Sensitivity: Distillation Temperature", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="lower right")

ax = axes[1]
ax.plot(alpha_df["Alpha"], alpha_df["best_val_acc"], "o-", color="#4363d8", linewidth=2, label="Val Accuracy (%)")
ax2 = ax.twinx()
ax2.plot(alpha_df["Alpha"], alpha_df["best_val_f1"] * 100, "s--", color="#e6194B", linewidth=2, label="Val F1 (weighted, %)")
ax.set_xlabel("Alpha (KD weight; 1-alpha = CE weight)"); ax.set_ylabel("Val Accuracy (%)", color="#4363d8")
ax2.set_ylabel("Val F1 weighted (%)", color="#e6194B")
ax.set_title("Sensitivity: KD Alpha (CE:KD Balance)", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="lower right")

ax = axes[2]
ax.plot([str(v) for v in lr_df["LR"]], lr_df["best_val_acc"], "o-", color="#4363d8", linewidth=2, label="Val Accuracy (%)")
ax2 = ax.twinx()
ax2.plot([str(v) for v in lr_df["LR"]], lr_df["best_val_f1"] * 100, "s--", color="#e6194B", linewidth=2, label="Val F1 (weighted, %)")
ax.set_xlabel("Learning Rate"); ax.set_ylabel("Val Accuracy (%)", color="#4363d8")
ax2.set_ylabel("Val F1 weighted (%)", color="#e6194B")
ax.set_title("Sensitivity: Student Learning Rate", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="lower right")

plt.suptitle("Hyperparameter Sensitivity Analysis — KD-CMAES Student (EfficientNet-B0 + CBAM)",
             fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sensitivity_analysis_all.png", dpi=600, bbox_inches="tight")
plt.show()

# ========================= COMBINED SUMMARY TABLE =========================
combined = pd.concat([
    temp_df.rename(columns={"Temperature": "Value"}).assign(Hyperparameter="Temperature (T)"),
    alpha_df.rename(columns={"Alpha": "Value"}).assign(Hyperparameter="KD Alpha"),
    lr_df.rename(columns={"LR": "Value"}).assign(Hyperparameter="Learning Rate"),
], ignore_index=True)
combined = combined[["Hyperparameter", "Value", "best_val_acc", "best_val_f1", "best_val_loss"]]
combined.columns = ["Hyperparameter", "Value", "Best Val Accuracy (%)", "Best Val F1 (weighted)", "Best Val Loss"]
combined.to_csv(f"{OUT_DIR}/sensitivity_analysis_combined.csv", index=False)

print("\n" + "=" * 70)
print("✅ SENSITIVITY ANALYSIS COMPLETE")
print("=" * 70)
print(combined.to_string(index=False))
print(f"\nAll files saved to: {OUT_DIR}")